# Embedding Evaluation

In [1]:
import numpy as np
import sys
import os

from pathlib import Path
sys.path.append(os.path.abspath('..'))

from src.skipgram import SkipGramModel,train_model
from src.negative_sampling import build_unigram_table
from src.tokenizer import Tokenizer

In [2]:
tok = Tokenizer()


with open("../data/corpus.txt", "r", encoding='utf-8') as f:
    raw_data = f.read()
    # Split the string by spaces and convert each "number" to an actual integer
    corpus_ids = [int(id_str) for id_str in raw_data.split()]
    
    
#tok.build_vocab(raw_data)


print(f"First 10 IDs: {corpus_ids[:10]}")

First 10 IDs: [29, 5, 2, 78, 1942, 45, 1064, 12, 101, 146]


In [3]:
unigram_table = build_unigram_table(corpus_ids,10000)

### Training

In [4]:
model = SkipGramModel(10000,100)

train_model(model,[corpus_ids[:500000]],unigram_table,10,5,0.025)

Epoch 1 | Pairs: 100000 | LR: 0.02500 | Loss: 3.68001
Epoch 1 | Pairs: 200000 | LR: 0.02500 | Loss: 3.30959
Epoch 1 | Pairs: 300000 | LR: 0.02500 | Loss: 3.11082
Epoch 1 | Pairs: 400000 | LR: 0.02500 | Loss: 2.98683
Epoch 1 | Pairs: 500000 | LR: 0.02500 | Loss: 2.89894
Epoch 1 | Pairs: 600000 | LR: 0.02500 | Loss: 2.83631
Epoch 1 | Pairs: 700000 | LR: 0.02500 | Loss: 2.78582
Epoch 1 | Pairs: 800000 | LR: 0.02500 | Loss: 2.74456
Epoch 1 | Pairs: 900000 | LR: 0.02500 | Loss: 2.70940
Epoch 1 | Pairs: 1000000 | LR: 0.02500 | Loss: 2.68138
Epoch 1 | Pairs: 1100000 | LR: 0.02500 | Loss: 2.65821
Epoch 1 | Pairs: 1200000 | LR: 0.02500 | Loss: 2.63809
Epoch 1 | Pairs: 1300000 | LR: 0.02500 | Loss: 2.62000
Epoch 1 | Pairs: 1400000 | LR: 0.02500 | Loss: 2.60238
Epoch 1 | Pairs: 1500000 | LR: 0.02500 | Loss: 2.58948
Epoch 1 | Pairs: 1600000 | LR: 0.02500 | Loss: 2.57781
Epoch 1 | Pairs: 1700000 | LR: 0.02500 | Loss: 2.56574
Epoch 1 | Pairs: 1800000 | LR: 0.02500 | Loss: 2.55721
Epoch 1 | Pairs: 19

FileNotFoundError: [Errno 2] No such file or directory: './data/embeddings.npy'

In [ ]:
def find_nearest_neighbors(query_word,word2idx,idx2word,normalized_matrix,top_k=5):
    if query_word not in word2idx:
        print(f"{query_word} is not in the dictionary")
        return
    
    
    #1 get the id and vector for the query word
    query_id = word2idx[query_word]
    query_vector = normalized_matrix[query_id]
    
    #2. cosine similarity 
    similarities = np.dot(normalized_matrix,query_vector)
    
    #3. sort  the indices based on similarity scores
    sorted_indices = np.argsort(similarities)
    
    #4. Grab the top k indices
    #remove the last index and reverse,
    best_indices = sorted_indices[-(top_k + 1):-1][::-1]
    
    print(f"words most similar to '{query_word}':")
    for idx in best_indices:
        print(f" {idx2word[id]} (Score: {similarities[idx]:.4f})")
            

In [ ]:
norms = np.linalg.norm(model.W1, axis=1,keepdim=True)

norms[norms == 0] = 1e-10

normalized_W1 = model.W1 / norms

idx2word = {idx: word for word, idx in tok.word_to_id.items()}